In [1]:
import os
import cv2
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, Activation, Input
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ==========================================
# 1. CONFIGURATION
# ==========================================
BASE_PATH = "./Datasets"
#######
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20

In [2]:
# ==========================================
# 2. PRÉTRAITEMENT (PAPIER)
# ==========================================
def crop_image_from_gray(img, tol=7):
    # Si l'image est en niveaux de gris
    if img.ndim == 2:
        # Crée un masque pour garder uniquement les pixels plus clairs que la tolérance (on enlève les bordures trop sombres)
        mask = img > tol
        # Recadre l'image en gardant uniquement les lignes/colonnes avec au moins un pixel > tolérance
        return img[np.ix_(mask.any(1), mask.any(0))]

    # Si l'image est en couleur (3 canaux : RGB)
    elif img.ndim == 3:
        # Convertit l'image en niveaux de gris pour repérer les bords sombres
        gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        mask = gray_img > tol  # Masque des pixels significatifs (plus clairs que la tolérance)

        # Vérifie si le recadrage laisserait une image vide
        check_shape = img[:,:,0][np.ix_(mask.any(1), mask.any(0))].shape[0]
        if (check_shape == 0):
            return img  # Si l'image est vide après recadrage, on retourne l'image d'origine
        else:
            # Applique le recadrage à chaque canal R, G, B
            img1 = img[:,:,0][np.ix_(mask.any(1), mask.any(0))]
            img2 = img[:,:,1][np.ix_(mask.any(1), mask.any(0))]
            img3 = img[:,:,2][np.ix_(mask.any(1), mask.any(0))]
            # Recompose l'image avec les 3 canaux recadrés
            return np.stack([img1, img2, img3], axis=-1)

def preprocess_image(path):
    try:
        # Charge l'image depuis le chemin donné
        img = cv2.imread(path)
        if img is None:
            return None  # Si le chemin est invalide ou l'image est vide, retourne None

        # Convertit l'image de BGR (format OpenCV) à RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Recadre l'image pour supprimer les bords noirs
        img = crop_image_from_gray(img)

        # Redimensionne l'image à une taille standard (définie par IMG_SIZE)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        # Convertit l'image RGB en espace de couleur LAB (séparation luminance et couleurs) puis sépare l'espace L des espaces A et B
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)

        # Applique CLAHE sur le canal L pour améliorer localement le contraste
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        cl = clahe.apply(l)

        # Recompose l'image LAB avec le L modifié (meilleur contraste)
        img = cv2.merge((cl, a, b))

        # Reconvertit en RGB après amélioration du contraste
        img = cv2.cvtColor(img, cv2.COLOR_LAB2RGB)

        # Normalise les pixels entre 0 et 1 pour entraînement
        return img.astype('float32') / 255.0

    except:
        # En cas d'erreur quelconque (image introuvable, mauvaise lecture, etc.)
        return None

In [3]:
# ==========================================
# 3. CHARGEMENT ET ÉTIQUETAGE (CORRIGÉ)
# ==========================================
def load_dataset_from_drive(dataset_name):
    print(f"\n Chargement de {dataset_name}...")

    # 1. Trouver le bon dossier racine
    target_dir = None
    for d in os.listdir(BASE_PATH):
        # Astuce pour trouver "DiaretDB1" même si c'est écrit "diaretdb1"
        if dataset_name.lower().replace("db","") in d.lower().replace("db",""):
            target_dir = os.path.join(BASE_PATH, d)
            break

    if not target_dir:
        print(f"Dossier introuvable : {dataset_name}")
        return np.array([]), np.array([])

    images = []
    labels = []
    seen_ids = set()

    # Regex pour trouver n'importe quel nombre dans le nom du fichier
    pattern_num = re.compile(r'(\d+)')

    # --- Cas spécial DRIMDB (Structure Good/Bad/Outlier) ---
    if "drim" in dataset_name.lower():
        for class_folder in ["Good", "Bad", "Outlier"]:
            class_path = os.path.join(target_dir, class_folder)

            if os.path.exists(class_path):
                # os.listdir ne rentre pas dans les sous-dossiers, c'est parfait ici.
                for file in os.listdir(class_path):
                    if file.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):

                        # Filtre anti-doublon strict
                        if file in seen_ids: continue
                        seen_ids.add(file)

                        # Label (Good = 0, Bad/Outlier = 1)
                        label = 0 if class_folder == "Good" else 1

                        # Chargement
                        img = preprocess_image(os.path.join(class_path, file))
                        if img is not None:
                            images.append(img)
                            labels.append(label)

    # --- CAS DIARETDB (Structure à plat ou sous-dossiers simples) ---
    else:
        for root, dirs, files in os.walk(target_dir):
            # On évite les dossiers techniques (annotations, masques)
            if any(x in root.lower() for x in ["ground", "mask", "dot", "eval"]):
                continue

            for file in files:
                if file.lower().endswith(('.png')):

                    # Extraction ID (ex: image015 -> 15)
                    match = pattern_num.findall(file)
                    if not match: continue
                    img_id = int(match[-1]) # Prend le dernier chiffre

                    # Anti-doublon par ID pour DiaretDB
                    if img_id in seen_ids: continue
                    seen_ids.add(img_id)

                    label = 1 # Par défaut Malade

                    # Règle DB0: 1 à 20 = Sain
                    if "db0" in dataset_name.lower() and img_id <= 20:
                        label = 0
                    # Règle DB1: Tout malade (Pas de sains dans ce dataset 89 images)

                    img = preprocess_image(os.path.join(root, file))
                    if img is not None:
                        images.append(img)
                        labels.append(label)

    X = np.array(images)
    y = np.array(labels)

    print(f"   -> {len(X)} images chargées.")
    # On vérifie que toutes les images soit bien recupéré
    if "db1" in dataset_name.lower() and len(X) < 89:
        missing = sorted(list(set(range(1, 90)) - seen_ids))
        print(f"Note: Images manquantes (IDs): {missing}")
    # on affiche notre echantillonage
    if len(X) > 0:
        print(f"   -> Sains: {len(y[y==0])} | Malades: {len(y[y==1])}")

    return X, y

In [5]:
def build_paper_cnn():
    model = Sequential([
        # ----- COUCHE D'ENTRÉE -----
        # L'image d'entrée a une taille IMG_SIZE x IMG_SIZE avec 3 canaux (RGB)
        Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

        # ----- BLOC CONVOLUTIONNEL 1 -----
        # Première couche de convolution : détecte des motifs simples (bords, textures...)
        Conv2D(8, (3, 3), strides=(1, 1), padding='same'),
        # Normalisation pour stabiliser l'apprentissage
        BatchNormalization(epsilon=1e-5),
        # Fonction d'activation : rend le modèle non linéaire
        Activation('relu'),
        # Réduction de la taille de l'image (downsampling), garde les infos importantes
        MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),

        # ----- BLOC CONVOLUTIONNEL 2 -----
        # Convolutions plus profondes : détectent des motifs plus complexes
        Conv2D(16, (3, 3), strides=(1, 1), padding='same'),
        BatchNormalization(epsilon=1e-5),
        Activation('relu'),
        MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),

        # ----- BLOC CONVOLUTIONNEL 3 -----
        # plus de filtres : le réseau apprend à repérer des structures spécifiques à l'œil,
        # comme les lésions, vaisseaux, opacités...
        Conv2D(32, (3, 3), strides=(1, 1), padding='same'),
        BatchNormalization(epsilon=1e-5),
        Activation('relu'),
        MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),

        # ----- COUCHE DE FIN -----
        # Transforme les cartes de caractéristiques en vecteur
        Flatten(),

        # Couche dense : combine toutes les caractéristiques pour décider la classe
        Dense(64, activation='relu'),
        # Évite le surapprentissage (overfitting) en désactivant 30% des neurones pendant l'entraînement
        Dropout(0.3),

        # Sortie : 2 classes
        Dense(2, activation='softmax')
    ])

    opt = SGD(learning_rate=0.01, momentum=0.9)
    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [6]:
# ==========================================
# 5. ENTRAÎNEMENT ET ÉVALUATION
# ==========================================
# Utilisation des noms exacts pour faciliter la correspondance
datasets_list = ["DiaretDB0", "DiaretDB1", "DRIMDB"]
results_store = {}

# Dictionnaire de référence avec des clés insensibles à la casse ou adaptées
refs = {
    "diaretdb0": 0.6267,
    "diaretdb1": 0.579,
    "drimdb": 0.68
}

# =============== BOUCLE SUR LES DATASETS ===============
for ds_name in datasets_list:

    # Chargement des images X et des labels y depuis Google Drive
    X, y = load_dataset_from_drive(ds_name)

    # Vérification que le dataset contient assez d’images
    if len(X) > 10:

        # Encodage One-Hot
        y_cat = to_categorical(y, num_classes=2)

        # Découpage : 80% entraînement | 20% test
        try:
            # stratify=y permet de conserver la même proportion de classes dans train et test
            X_train, X_test, y_train, y_test = train_test_split(
                X, y_cat, test_size=0.2, random_state=42, stratify=y
            )
        except:
            # Si stratify échoue (déséquilibre), on retire simplement l’option
            X_train, X_test, y_train, y_test = train_test_split(
                X, y_cat, test_size=0.2, random_state=42
            )

        print(f"   Entraînement sur {ds_name} (20 epochs strict)...")

        # Construction du modèle CNN
        model = build_paper_cnn()

        # Callback : réduit automatiquement le learning rate si la val_loss stagne
        callbacks = [
            ReduceLROnPlateau(
                monitor='val_loss', factor=0.2, patience=5, min_lr=0.00001
            )
        ]

        # Entraînement du réseau
        history = model.fit(
            X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=EPOCHS, batch_size=BATCH_SIZE,
            verbose=1,
            callbacks=callbacks
        )

        # Évaluation sur l'ensemble de test
        loss, acc = model.evaluate(X_test, y_test, verbose=0)

        # Stockage de la précision dans un dictionnaire
        results_store[ds_name.lower()] = acc

        print(f"Précision finale : {acc:.2%}")

        # Prédictions du modèle (converties en labels 0/1)
        y_pred = np.argmax(model.predict(X_test), axis=1)
        y_true = np.argmax(y_test, axis=1)

        # Affichage d’un rapport complet (precision, recall, F1-score)
        print(
            classification_report(
                y_true, y_pred,
                labels=[0, 1],
                target_names=['Sain', 'Malade'],
                zero_division=0
            )
        )

    else:
        # Dataset trop petit → impossible de l'entraîner correctement
        print(f"Données insuffisantes.")
        results_store[ds_name.lower()] = 0.0


# ==========================================
# 6. TABLEAU FINAL CORRIGÉ
# ==========================================
print("\n\n TABLEAU DE RÉSULTATS")
print("-" * 60)
print(f"{'Dataset':<15} | {'Papier (Ref)':<15} | {'Votre CNN':<15}")
print("-" * 60)

# Affichage en utilisant les clés minuscules pour garantir la correspondance
for ds in datasets_list:
    key = ds.lower()
    score = results_store.get(key, 0.0)
    ref = refs.get(key, 0.0)
    print(f" {ds:<15} | {ref:.1%}           | {score:.1%}")
print("-" * 60)


 Chargement de DiaretDB0...
   -> 130 images chargées.
   -> Sains: 20 | Malades: 110
   Entraînement sur DiaretDB0 (20 epochs strict)...
Epoch 1/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 177ms/step - accuracy: 0.8077 - loss: 3.2086 - val_accuracy: 0.8462 - val_loss: 0.4478 - learning_rate: 0.0100
Epoch 2/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 133ms/step - accuracy: 0.6635 - loss: 1.3029 - val_accuracy: 0.8462 - val_loss: 0.4468 - learning_rate: 0.0100
Epoch 3/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 132ms/step - accuracy: 0.6442 - loss: 0.9756 - val_accuracy: 0.1538 - val_loss: 3.8598 - learning_rate: 0.0100
Epoch 4/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 134ms/step - accuracy: 0.6538 - loss: 1.3143 - val_accuracy: 0.1538 - val_loss: 1.3074 - learning_rate: 0.0100
Epoch 5/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 131ms/step - accuracy: 0.8462 - loss: 0.5686 - val_accuracy: 0.6923 - val_loss: 0.5981 - learning_rate: 0.0100
Epoch 6/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 129ms/step - accuracy: 0.8462 - loss: 0.4845 - val_accuracy: 0.2692 - va